# R Master v3C · 断点续跑版（手机专用）

这版是为 **Colab/iPhone 反复断线** 重构的，不再让你从头重跑。

只做：
1. 打开链接；
2. 点 **运行全部 / Run all**；
3. 允许 Google Drive。

它会把每一张已经完成的渲染图 **立刻写进 Google Drive**。  
哪怕中途断线，重新打开再点一次“运行全部”，已经完成的步骤会自动跳过，从断点继续。

它直接读取：
`MyDrive/R_Master/v2/latest/R_Master_Align_v2_PREVIEW.blend`

不需要上传 Mona，也不重复做 v2 比例计算。


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, subprocess, os, textwrap, json, zipfile
print("R Master v3C · 断点续跑版")
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/R_Master")
V2=ROOT/"v2"/"latest"/"R_Master_Align_v2_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v3_checkpoint"/"latest"
CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
if not V2.exists():
    raise RuntimeError("没找到 v2 预览文件，请把这屏截图给二蛋。")
print("✓ Drive 已挂载")
print(f"✓ v2 源：{V2.stat().st_size/1024/1024:.1f} MiB")



In [ ]:
from pathlib import Path
import shutil, subprocess, os
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v3c")
LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender Drive 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)
    print("✓ Blender 已补充缓存")

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender 就绪")



In [ ]:
from pathlib import Path
SCRIPT=LOCAL/"R_Master_v3_checkpoint_render.py"
SCRIPT.write_text("\nimport bpy, os, sys, json\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None; view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\": out=argv[i+1]\n    if a==\"--view\": view=argv[i+1]\nif not out or not view:\n    raise RuntimeError(\"missing --out/--view\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\") or bpy.data.objects.get(\"Mona_Main\")\nif not body or body.type!=\"MESH\":\n    raise RuntimeError(\"body mesh missing\")\n\nfor obj in bpy.context.scene.objects:\n    if obj.type==\"MESH\":\n        obj.hide_render=(obj!=body)\n        obj.hide_viewport=(obj!=body)\n    elif obj.type==\"ARMATURE\":\n        obj.hide_render=True\n\nfor mod in body.modifiers:\n    if mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\"} or \"mask\" in mod.name.lower() or \"cloth\" in mod.name.lower():\n        mod.show_viewport=False\n        mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1)\n        mod.render_levels=min(mod.render_levels,1)\n\npts=[body.matrix_world@Vector(c) for c in body.bound_box]\nmn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\nmx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\ncenter=(mn+mx)*.5\nheight=mx.z-mn.z; width=mx.x-mn.x; depth=mx.y-mn.y\ndist=max(height,width,depth)*2.5\n\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_WORKBENCH\"\nscene.render.image_settings.file_format=\"PNG\"\nscene.render.film_transparent=False\nscene.display.shading.light=\"STUDIO\"\nscene.display.shading.show_shadows=True\nscene.display.shading.show_cavity=True\nscene.display.shading.cavity_type=\"WORLD\"\nscene.display.shading.color_type=\"SINGLE\"\nscene.display.shading.single_color=(0.58,0.58,0.61)\nscene.display.shading.background_type=\"VIEWPORT\"\nscene.display.shading.background_color=(0.045,0.045,0.055)\n\ncam_data=bpy.data.cameras.get(\"R_Master_v3C_Camera_DATA\") or bpy.data.cameras.new(\"R_Master_v3C_Camera_DATA\")\ncam=bpy.data.objects.get(\"R_Master_v3C_Camera\")\nif not cam:\n    cam=bpy.data.objects.new(\"R_Master_v3C_Camera\",cam_data)\n    scene.collection.objects.link(cam)\nscene.camera=cam\ncam.data.type=\"ORTHO\"\n\ndef look_at(obj,target):\n    obj.rotation_euler=(Vector(target)-obj.location).to_track_quat(\"-Z\",\"Y\").to_euler()\n\ndef render(path,pos,target,scale,res):\n    scene.render.resolution_x,scene.render.resolution_y=res\n    scene.render.resolution_percentage=100\n    cam.location=Vector(pos)\n    cam.data.ortho_scale=scale\n    look_at(cam,Vector(target))\n    scene.render.filepath=path\n    bpy.ops.render.render(write_still=True)\n\nspec={}\nspec[\"full_front\"]=(\"R_Master_v3_full_front.png\",(center.x,center.y-dist,center.z),center,height*1.08,(720,960))\nspec[\"full_side\"]=(\"R_Master_v3_full_side.png\",(center.x+dist,center.y,center.z),center,height*1.08,(720,960))\nspec[\"full_three_quarter\"]=(\"R_Master_v3_full_three_quarter.png\",(center.x+dist*.72,center.y-dist*.72,center.z),center,height*1.08,(720,960))\nfor key,frac in [(\"torso_waist\",.61),(\"waist_pelvis\",.51),(\"pelvis_upperthigh\",.42)]:\n    z=mn.z+height*frac\n    spec[key]=(f\"R_Master_v3_{key}.png\",(center.x,center.y-dist,z),(center.x,center.y,z),max(.42,height*.25),(900,700))\nz=mn.z+height*.46\nspec[\"glute_side\"]=(\"R_Master_v3_glute_side.png\",(center.x+dist,center.y,z),(center.x,center.y,z),max(.42,height*.25),(900,700))\n\nif view not in spec:\n    raise RuntimeError(\"unknown view \"+str(view))\nfname,pos,target,scale,res=spec[view]\npath=os.path.join(out,fname)\nrender(path,pos,target,scale,res)\nprint(\"[R Master v3C] RENDER_OK\",view,path)\n\n# lightweight report updated after every completed view\nreport_path=os.path.join(out,\"R_Master_v3_checkpoint_report.json\")\ndone=sorted([k for k,(f,*_) in spec.items() if os.path.exists(os.path.join(out,f))])\nwith open(report_path,\"w\",encoding=\"utf-8\") as f:\n    json.dump({\n        \"ok\":True,\n        \"stage\":\"R_Master_v3_checkpoint\",\n        \"body_mesh\":body.name,\n        \"completed_views\":done,\n        \"total_views\":len(spec),\n        \"rest_pose_baked\":False,\n        \"final_vrm\":False\n    },f,ensure_ascii=False,indent=2)\n",encoding="utf-8")
print("✓ 断点渲染脚本就绪")



In [ ]:
from pathlib import Path
import subprocess, os, json, time
views=[
 ("full_front","R_Master_v3_full_front.png"),
 ("full_side","R_Master_v3_full_side.png"),
 ("full_three_quarter","R_Master_v3_full_three_quarter.png"),
 ("torso_waist","R_Master_v3_torso_waist.png"),
 ("waist_pelvis","R_Master_v3_waist_pelvis.png"),
 ("pelvis_upperthigh","R_Master_v3_pelvis_upperthigh.png"),
 ("glute_side","R_Master_v3_glute_side.png"),
]
for idx,(view,fname) in enumerate(views,1):
    dest=OUT/fname
    if dest.exists() and dest.stat().st_size>20_000:
        print(f"✓ [{idx}/7] {view} 已存在，跳过")
        continue
    print(f"▶ [{idx}/7] 正在渲染 {view}…")
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(V2),"--python",str(SCRIPT),"--","--out",str(OUT),"--view",view]
    p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    print("\n".join([x for x in p.stdout.splitlines() if "R Master v3C" in x or "Error" in x or "Traceback" in x][-10:]))
    if p.returncode!=0:
        raise RuntimeError(f"{view} 渲染失败，退出码 {p.returncode}。截图给二蛋即可。")
    if not dest.exists():
        raise RuntimeError(f"{view} 没有生成输出。")
    print(f"✓ [{idx}/7] 已写入 Drive：{fname}")
print("✓ 7 张图全部完成")



In [ ]:
from IPython.display import display,Image,Markdown
from pathlib import Path
import zipfile, json, shutil
views=[
 ("全身正面","R_Master_v3_full_front.png"),
 ("全身侧面","R_Master_v3_full_side.png"),
 ("全身 3/4","R_Master_v3_full_three_quarter.png"),
 ("胸廓→腰","R_Master_v3_torso_waist.png"),
 ("腰→骨盆","R_Master_v3_waist_pelvis.png"),
 ("骨盆→大腿根","R_Master_v3_pelvis_upperthigh.png"),
 ("臀线侧视","R_Master_v3_glute_side.png"),
]
for title,fname in views:
    p=OUT/fname
    if p.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(p),width=500))
review=OUT/"R_Master_v3_Review.zip"
if review.exists(): review.unlink()
with zipfile.ZipFile(review,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for _,fname in views:
        p=OUT/fname
        if p.exists(): z.write(p,arcname=p.name)
    rp=OUT/"R_Master_v3_checkpoint_report.json"
    if rp.exists(): z.write(rp,arcname=rp.name)
print(f"✓ Review ZIP：{review.stat().st_size/1024/1024:.1f} MiB")
files.download(str(review))

